<a href="https://colab.research.google.com/github/tekpinar/gromacscolab/blob/main/MD_EnergyAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GROMACS Installation and Environment Verification
This cell installs the GROMACS package and verifies the installation by displaying the program version, ensuring the simulation environment is correctly configured.

In [ ]:
#@title Install GROMACS
!apt-get update -qq
!apt-get install -y -qq gromacs
!pip install -q plotly kaleido==0.2.1
import subprocess
print(subprocess.run(['gmx','--version'],capture_output=True,text=True).stdout.splitlines()[0])


# Uploading Input Files to the Working Environment
This cell uploads files from the local computer to the working environment so they can be used as input for the simulation or analysis steps. At this stage, please upload the .edr file generated from the simulation for energy analysis.

In [ ]:
from google.colab import files
uploaded_edr = files.upload()
edr_filename = list(uploaded_edr.keys())[0]




## **Select Atom Type**
This cell allows the user to select a specific energy term from the GROMACS energy file. The chosen quantity is automatically mapped to its corresponding index number so it can be extracted during analysis.

Different energy terms describe different physical properties of the system. Potential energy is used to evaluate overall stability, temperature verifies thermostat performance, and pressure indicates whether the system has reached equilibrium.

In [ ]:
#@title Read the energy terms that are actually present in this .edr
import subprocess, ipywidgets as widgets
from IPython.display import display

# `gmx energy` prints its term table and exits when given empty input.
p = subprocess.run(['gmx','energy','-f',edr_filename,'-o','/dev/null'],
                   input='', capture_output=True, text=True)

terms, seen = [], set()
for line in (p.stdout + p.stderr).splitlines():
    s = line.strip()
    if not s or s[0].isdigit() is False:
        continue
    # table rows look like:  1  Bond   2  Angle   3  Proper-Dih.  ...
    toks = s.split()
    for i in range(0, len(toks) - 1, 2):
        if toks[i].isdigit():
            name = toks[i + 1]
            if name not in seen:
                seen.add(name); terms.append(name)

assert terms, "Could not read energy terms from " + edr_filename
print(f"{len(terms)} terms available in {edr_filename}:")
print("   " + ", ".join(terms))

default = [t for t in ('Potential','Temperature','Pressure','Density',
                       'Total-Energy','Volume') if t in terms] or terms[:4]

term_widget = widgets.SelectMultiple(options=terms, value=tuple(default),
                                     rows=min(16, len(terms)),
                                     description='Terms:',
                                     style={'description_width':'initial'})
display(term_widget)
print("\nCtrl/Cmd-click to select several terms, then run the next cell.")


# Energy File Definition and Term Identification
This cell defines the energy file to be analyzed and displays the selected energy term along with its corresponding index. Printing these values ensures that the correct energy component from the simulation output will be extracted in the subsequent analysis step.

Verifying the selected file and energy term helps prevent errors and confirms that the intended physical quantity is being analyzed.

In [ ]:
energy_file = edr_filename
selected_terms = list(term_widget.value)
print("Energy file   :", energy_file)
print("Selected terms:", selected_terms)
assert selected_terms, "Select at least one term above."


# Extraction of Selected Energy Term and XVG File Generation
This cell extracts the selected energy term from the GROMACS energy file and saves it as an .xvg output file. The chosen quantity (such as potential energy, kinetic energy, temperature, or pressure) is written as a time-dependent dataset that can be plotted and analyzed.

The resulting file will be used to generate graphs and evaluate the stability and thermodynamic behavior of the molecular dynamics simulation.

In [ ]:
import subprocess, re, os

# Filenames are sanitised: term names such as "LJ-(SR)" and "#Surf*SurfTen"
# contain shell metacharacters, and the previous f-string built them straight
# into a `!echo ... | gmx energy` command line.
def safe(name):
    return re.sub(r'[^A-Za-z0-9._-]', '_', name)

xvg_files = {}
for term in selected_terms:
    out = f"energy_{safe(term)}.xvg"
    p = subprocess.run(['gmx','energy','-f',energy_file,'-o',out],
                       input=term + "\n\n", capture_output=True, text=True)
    if os.path.exists(out):
        xvg_files[term] = out
        stats = [l for l in (p.stdout+p.stderr).splitlines() if l.startswith(term)]
        print(f"{term:<18} -> {out}")
        if stats:
            print(f"   {stats[-1].strip()}")
    else:
        print(f"{term:<18} -> FAILED")

assert xvg_files, "No energy terms could be extracted."


# Visualization and Analysis of Energy Time Series
In this cell, the selected energy term is read from the .xvg file and plotted as a function of time. The data are converted into an analyzable format and visualized interactively.

To reduce short-term fluctuations and highlight the overall trend, a moving average is applied. The window size (window_size) defines the number of data points used in the averaging and is set to 100 by default. Smaller values preserve fluctuations, while larger values provide stronger smoothing.

This plot is used to evaluate the equilibration and stability of the simulation. A stable system exhibits fluctuations around a constant mean, whereas a noticeable drift indicates that equilibrium has not yet been reached.

In [ ]:
#@title Plot the selected terms with a running mean
window_size = 100  #@param {type:"integer"}

import numpy as np, plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_xvg(path):
    x, y, xlab, ylab = [], [], 'Time (ps)', ''
    for line in open(path):
        if line.startswith('@'):
            if 'xaxis  label' in line: xlab = line.split('"')[1]
            elif 'yaxis  label' in line: ylab = line.split('"')[1]
        elif not line.startswith(('#','&')):
            p = line.split()
            if len(p) >= 2:
                x.append(float(p[0])); y.append(float(p[1]))
    return np.array(x), np.array(y), xlab, ylab

terms = list(xvg_files)
fig = make_subplots(rows=len(terms), cols=1, shared_xaxes=True,
                    subplot_titles=terms, vertical_spacing=0.06)

summary = []
for i, term in enumerate(terms, start=1):
    x, y, xlab, ylab = read_xvg(xvg_files[term])
    w = max(1, min(window_size, len(y)))
    run = np.convolve(y, np.ones(w)/w, mode='valid')
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=term,
                             line=dict(width=1), opacity=0.45,
                             showlegend=False), row=i, col=1)
    fig.add_trace(go.Scatter(x=x[w-1:], y=run, mode='lines',
                             name=f'{term} (mean {w})',
                             line=dict(width=2), showlegend=False), row=i, col=1)
    fig.update_yaxes(title_text=ylab or term, row=i, col=1)
    half = len(y)//2
    summary.append((term, y.mean(), y.std(), y[half:].mean(), y[half:].std()))

fig.update_xaxes(title_text=xlab, row=len(terms), col=1)
fig.update_layout(height=260*len(terms), hovermode='x unified',
                  title='Thermodynamic properties over the simulation')
fig.show()
fig.write_image("energy_summary.png", scale=2)

print(f"{'Term':<18}{'mean(all)':>14}{'sd(all)':>12}{'mean(2nd half)':>17}{'sd':>12}")
for t, m, s, m2, s2 in summary:
    print(f"{t:<18}{m:>14.3f}{s:>12.3f}{m2:>17.3f}{s2:>12.3f}")
print("\nA property is equilibrated when the second-half mean matches the overall\n"
      "mean to within its standard deviation and shows no systematic drift.")
